# 03 — Model Architecture & Training Utilities

**Project:** Tomato Quality Semantic Segmentation — Background Bias Analysis  
**Models:** U-Net with MobileNetV2 & EfficientNet-B0 Encoders  

## What this notebook does

1. Defines the U-Net model factory (`create_unet_model`) for both encoders  
2. Defines `CombinedLoss` — Weighted Cross-Entropy + Dice Loss  
3. Defines all evaluation metrics: pixel accuracy, per-class IoU, Dice, confusion matrix  
4. Defines calibration utilities: ECE computation and reliability diagrams  
5. Defines the full training loop with early stopping on validation mIoU  
6. Defines all visualization utilities: training curves, prediction grids, confusion matrices  
7. Defines Grad-CAM utilities for explainability analysis  
8. Runs a quick smoke-test on both model architectures to verify everything works  

> **Nothing is trained here.** This notebook only defines and validates the infrastructure.  
> Actual training happens in notebooks 04 (Step 1), 05 (Step 2), and 06 (Step 3).

## 1. Imports

In [1]:
import os
import sys
import json
import random
import warnings
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import segmentation_models_pytorch as smp
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi']    = 100

print('Imports OK')

Imports OK


## 2. Load Shared Config

In [2]:
sys.path.insert(0, str(Path('..').resolve()))
from config import *

print(f'Device      : {DEVICE}')
print(f'NUM_CLASSES : {NUM_CLASSES}')
print(f'IMG_SIZE    : {IMG_SIZE}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')

Device      : cpu
NUM_CLASSES : 7
IMG_SIZE    : 512
OUTPUT_DIR  : C:\Personal Data\Thesis\tomato-background-bias\outputs


## 3. Model Factory

In [3]:
def create_unet_model(encoder_name, num_classes=NUM_CLASSES, pretrained='imagenet'):
    """
    Create a U-Net model with a given encoder.

    Supported encoders used in this project:
      - 'mobilenet_v2'   : MobileNetV2 (lightweight)
      - 'efficientnet-b0': EfficientNet-B0 (efficiency-accuracy trade-off)

    Args:
        encoder_name : SMP encoder string
        num_classes  : number of output segmentation classes (default 7)
        pretrained   : encoder weight initialisation ('imagenet' or None)

    Returns:
        model on DEVICE
    """
    model = smp.Unet(
        encoder_name    = encoder_name,
        encoder_weights = pretrained,
        in_channels     = 3,
        classes         = num_classes,
        activation      = None,   # raw logits — loss handles softmax
    )
    return model.to(DEVICE)


print('create_unet_model() defined.')

create_unet_model() defined.


### 3.1 Architecture Summary

In [4]:
for enc_name, label in [('mobilenet_v2', 'MobileNetV2'), ('efficientnet-b0', 'EfficientNet-B0')]:
    m = create_unet_model(enc_name)
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{label}')
    print(f'  Total parameters     : {total:>12,}')
    print(f'  Trainable parameters : {trainable:>12,}')
    del m
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

MobileNetV2
  Total parameters     :    6,629,815
  Trainable parameters :    6,629,815


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet-B0
  Total parameters     :    6,252,339
  Trainable parameters :    6,252,339


## 4. Loss Function — Weighted CE + Dice

In [5]:
class CombinedLoss(nn.Module):
    """
    Weighted Cross-Entropy + Dice Loss for semantic segmentation.

    Combines two complementary losses:
    - Cross-Entropy   : pixel-wise classification with class weights for imbalance
    - Dice Loss       : region-overlap metric, robust to class imbalance

    Args:
        class_weights : FloatTensor of shape (num_classes,) for CE weighting
        dice_weight   : weight for Dice component (default 0.5)
        ce_weight     : weight for CE component   (default 0.5)
        num_classes   : number of segmentation classes
        smooth        : Laplace smoothing for Dice (avoids division by zero)
    """

    def __init__(self, class_weights=None, dice_weight=0.5, ce_weight=0.5,
                 num_classes=NUM_CLASSES, smooth=1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight   = ce_weight
        self.num_classes = num_classes
        self.smooth      = smooth

        self.ce_loss = nn.CrossEntropyLoss(
            weight=class_weights) if class_weights is not None else nn.CrossEntropyLoss()

    def dice_loss(self, pred, target):
        """
        Multi-class Dice loss.
        pred   : (B, C, H, W) logits
        target : (B, H, W) class indices
        """
        pred_soft     = F.softmax(pred, dim=1)                          # (B, C, H, W)
        target_one_hot = F.one_hot(target, self.num_classes)            # (B, H, W, C)
        target_one_hot = target_one_hot.permute(0, 3, 1, 2).float()    # (B, C, H, W)

        intersection   = (pred_soft * target_one_hot).sum(dim=(2, 3))
        cardinality    = pred_soft.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)

        return 1.0 - dice_per_class.mean()

    def forward(self, pred, target):
        ce   = self.ce_loss(pred, target)
        dice = self.dice_loss(pred, target)
        return self.ce_weight * ce + self.dice_weight * dice


print('CombinedLoss (Weighted CE + Dice) defined.')

CombinedLoss (Weighted CE + Dice) defined.


## 5. Evaluation Metrics

In [6]:
def compute_metrics(pred_masks, gt_masks, num_classes=NUM_CLASSES):
    """
    Compute segmentation metrics from lists of predicted and ground-truth masks.

    Args:
        pred_masks : list of np.ndarray (H, W) — predicted class indices
        gt_masks   : list of np.ndarray (H, W) — ground truth class indices

    Returns:
        dict with keys:
          pixel_accuracy, mean_iou, mean_dice,
          iou_per_class, dice_per_class, acc_per_class,
          confusion_matrix, class_present
    """
    if not pred_masks or not gt_masks:
        return {
            'pixel_accuracy' : 0.0,
            'mean_iou'       : 0.0,
            'mean_dice'      : 0.0,
            'iou_per_class'  : np.zeros(num_classes),
            'dice_per_class' : np.zeros(num_classes),
            'acc_per_class'  : np.zeros(num_classes),
            'confusion_matrix': np.zeros((num_classes, num_classes)),
            'class_present'  : np.zeros(num_classes, dtype=bool),
        }

    all_pred = np.concatenate([m.flatten() for m in pred_masks])
    all_gt   = np.concatenate([m.flatten() for m in gt_masks])

    pixel_acc      = (all_pred == all_gt).mean()
    iou_per_class  = np.zeros(num_classes)
    dice_per_class = np.zeros(num_classes)
    acc_per_class  = np.zeros(num_classes)
    class_present  = np.zeros(num_classes, dtype=bool)

    for c in range(num_classes):
        pred_c   = (all_pred == c)
        gt_c     = (all_gt   == c)
        pred_sum = pred_c.sum()
        gt_sum   = gt_c.sum()

        if gt_sum == 0 and pred_sum == 0:
            iou_per_class[c]  = float('nan')
            dice_per_class[c] = float('nan')
            acc_per_class[c]  = float('nan')
            continue

        class_present[c]  = True
        intersection      = (pred_c & gt_c).sum()
        union             = (pred_c | gt_c).sum()

        iou_per_class[c]  = intersection / union             if union > 0           else 0.0
        dice_per_class[c] = 2 * intersection / (pred_sum + gt_sum) if pred_sum + gt_sum > 0 else 0.0
        acc_per_class[c]  = intersection / gt_sum            if gt_sum > 0          else 0.0

    valid_ious  = iou_per_class [class_present]
    valid_dices = dice_per_class[class_present]
    mean_iou    = np.nanmean(valid_ious)  if len(valid_ious)  > 0 else 0.0
    mean_dice   = np.nanmean(valid_dices) if len(valid_dices) > 0 else 0.0

    cm = confusion_matrix(all_gt, all_pred, labels=list(range(num_classes)))

    return {
        'pixel_accuracy'  : pixel_acc,
        'mean_iou'        : mean_iou,
        'mean_dice'       : mean_dice,
        'iou_per_class'   : iou_per_class,
        'dice_per_class'  : dice_per_class,
        'acc_per_class'   : acc_per_class,
        'confusion_matrix': cm,
        'class_present'   : class_present,
    }


def print_metrics(metrics, title='Evaluation Results'):
    """Pretty-print evaluation metrics."""
    print(f"\n{'='*60}")
    print(f'  {title}')
    print(f"{'='*60}")
    print(f"  Overall Pixel Accuracy : {metrics['pixel_accuracy']:.4f}")
    print(f"  Mean IoU               : {metrics['mean_iou']:.4f}")
    print(f"  Mean Dice              : {metrics['mean_dice']:.4f}")
    print(f"\n  {'Class':<20} {'IoU':>8} {'Dice':>8} {'Accuracy':>10}")
    print(f"  {'-'*48}")
    for c in range(NUM_CLASSES):
        iou  = metrics['iou_per_class'][c]
        dice = metrics['dice_per_class'][c]
        acc  = metrics['acc_per_class'][c]
        iou_str  = f'{iou:.4f}'  if not np.isnan(iou)  else '   N/A'
        dice_str = f'{dice:.4f}' if not np.isnan(dice) else '   N/A'
        acc_str  = f'{acc:.4f}'  if not np.isnan(acc)  else '   N/A'
        marker   = ' *' if not metrics['class_present'][c] else ''
        print(f"  {CLASS_NAMES[c]:<20} {iou_str:>8} {dice_str:>8} {acc_str:>10}{marker}")
    print(f"{'='*60}")
    absent = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if not metrics['class_present'][i]]
    if absent:
        print(f'  * Not present in eval set: {", ".join(absent)}')
    print(f"{'='*60}")


print('Evaluation metric functions defined.')

Evaluation metric functions defined.


## 6. Calibration Utilities

In [7]:
def compute_calibration(pred_probs_list, gt_masks_list, num_bins=15):
    """
    Compute Expected Calibration Error (ECE) and reliability diagram data.

    Args:
        pred_probs_list : list of np.ndarray (H, W, C) softmax probabilities
        gt_masks_list   : list of np.ndarray (H, W) ground truth class indices
        num_bins        : number of confidence bins

    Returns:
        dict with: ece, bin_confidences, bin_accuracies, bin_counts, bin_boundaries
    """
    all_conf    = []
    all_correct = []

    for probs, gt in zip(pred_probs_list, gt_masks_list):
        pred_class = probs.argmax(axis=-1)
        max_conf   = probs.max(axis=-1)
        correct    = (pred_class == gt).astype(np.float32)
        all_conf   .append(max_conf.flatten())
        all_correct.append(correct.flatten())

    all_conf    = np.concatenate(all_conf)
    all_correct = np.concatenate(all_correct)

    bin_boundaries  = np.linspace(0, 1, num_bins + 1)
    bin_confidences = np.zeros(num_bins)
    bin_accuracies  = np.zeros(num_bins)
    bin_counts      = np.zeros(num_bins)

    for i in range(num_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask   = (all_conf > lo) & (all_conf <= hi)
        if mask.sum() > 0:
            bin_confidences[i] = all_conf[mask].mean()
            bin_accuracies[i]  = all_correct[mask].mean()
            bin_counts[i]      = mask.sum()

    total = bin_counts.sum()
    ece   = np.sum(bin_counts / total * np.abs(bin_accuracies - bin_confidences))

    return {
        'ece'             : ece,
        'bin_confidences' : bin_confidences,
        'bin_accuracies'  : bin_accuracies,
        'bin_counts'      : bin_counts,
        'bin_boundaries'  : bin_boundaries,
    }


def plot_reliability_diagram(cal_data, title='Reliability Diagram', ax=None):
    """Plot a reliability diagram from calibration data."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(7, 7))

    bins   = cal_data['bin_confidences']
    accs   = cal_data['bin_accuracies']
    counts = cal_data['bin_counts']
    ece    = cal_data['ece']
    mask   = counts > 0

    ax.bar(bins[mask], accs[mask], width=1.0 / len(bins),
           alpha=0.6, color='steelblue', edgecolor='navy', label='Actual accuracy')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
    ax.set_xlabel('Mean Predicted Confidence', fontsize=12)
    ax.set_ylabel('Fraction of Correct Predictions', fontsize=12)
    ax.set_title(f'{title}\nECE = {ece:.4f}', fontsize=13)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=11)
    ax.set_aspect('equal')
    return ax


print('Calibration utilities defined.')

Calibration utilities defined.


## 7. Training & Evaluation Loops

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device=DEVICE):
    """Train for one epoch. Returns average loss."""
    model.train()
    running_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device=DEVICE, return_predictions=False):
    """
    Evaluate model on a DataLoader.

    Returns:
        avg_loss, metrics_dict
        If return_predictions=True: avg_loss, metrics_dict, pred_masks, gt_masks, pred_probs
    """
    model.eval()
    running_loss   = 0.0
    all_pred_masks = []
    all_gt_masks   = []
    all_pred_probs = []

    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        outputs      = model(imgs)
        running_loss += criterion(outputs, masks).item() * imgs.size(0)

        probs    = F.softmax(outputs, dim=1)
        preds    = probs.argmax(dim=1)
        preds_np = preds.cpu().numpy()
        masks_np = masks.cpu().numpy()
        probs_np = probs.cpu().numpy()

        for b in range(preds_np.shape[0]):
            all_pred_masks.append(preds_np[b])
            all_gt_masks  .append(masks_np[b])
            if return_predictions:
                all_pred_probs.append(probs_np[b].transpose(1, 2, 0))

    avg_loss = running_loss / len(loader.dataset)
    metrics  = compute_metrics(all_pred_masks, all_gt_masks)

    if return_predictions:
        return avg_loss, metrics, all_pred_masks, all_gt_masks, all_pred_probs
    return avg_loss, metrics


def train_model(model, train_loader, val_loader, test_loader,
                criterion, optimizer, scheduler,
                num_epochs=NUM_EPOCHS, patience=EARLY_STOP_PATIENCE,
                model_name='model', device=DEVICE):
    """
    Full training loop with early stopping on validation mIoU.
    Test set is evaluated ONCE at the very end using the best val checkpoint.

    Returns:
        model (best weights loaded), history dict, final_test_metrics dict
    """
    best_miou        = 0.0
    best_epoch       = 0
    best_state       = None
    epochs_no_improve = 0

    history = {
        'train_loss'     : [], 'val_loss'      : [], 'test_loss': [],
        'train_miou'     : [], 'val_miou'      : [],
        'train_pixel_acc': [], 'val_pixel_acc' : [],
    }

    print(f"{'='*60}")
    print(f'  Training: {model_name}')
    print(f"{'='*60}")

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        train_loss               = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_metrics    = evaluate(model, val_loader,   criterion, device)
        _,        train_metrics  = evaluate(model, train_loader, criterion, device)

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history['train_loss']     .append(train_loss)
        history['val_loss']       .append(val_loss)
        history['train_miou']     .append(train_metrics['mean_iou'])
        history['val_miou']       .append(val_metrics['mean_iou'])
        history['train_pixel_acc'].append(train_metrics['pixel_accuracy'])
        history['val_pixel_acc']  .append(val_metrics['pixel_accuracy'])

        elapsed = time.time() - t0

        if val_metrics['mean_iou'] > best_miou:
            best_miou         = val_metrics['mean_iou']
            best_epoch        = epoch
            best_state        = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            marker            = '  *** BEST ***'
        else:
            epochs_no_improve += 1
            marker             = ''

        print(f'  Epoch {epoch:3d}/{num_epochs} | '
              f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
              f'Train mIoU: {train_metrics["mean_iou"]:.4f} | '
              f'Val mIoU: {val_metrics["mean_iou"]:.4f} | '
              f'Time: {elapsed:.1f}s{marker}')

        if epochs_no_improve >= patience:
            print(f'\n  Early stopping at epoch {epoch} '
                  f'(no improvement for {patience} epochs)')
            break

    # ── Restore best checkpoint ──────────────────────────────────────────────
    if best_state is not None:
        model.load_state_dict(best_state)

    save_path = OUTPUT_DIR / f'{model_name}_best.pth'
    torch.save(best_state, save_path)
    print(f'\n  Best Val mIoU: {best_miou:.4f} at epoch {best_epoch}')
    print(f'  Model saved → {save_path}')

    # ── Final test evaluation — ONCE ONLY ────────────────────────────────────
    print('  Evaluating on TEST SET (first and only time)...')
    test_loss, final_test_metrics = evaluate(model, test_loader, criterion, device)
    history['test_loss'].append(test_loss)
    print(f'  Final Test mIoU: {final_test_metrics["mean_iou"]:.4f}  |  '
          f'Test Pixel Acc: {final_test_metrics["pixel_accuracy"]:.4f}')

    return model, history, final_test_metrics


print('Training & evaluation loops defined.')

Training & evaluation loops defined.


## 8. Visualization Utilities

In [9]:
def plot_training_history(history, model_name='Model'):
    """Plot training curves: loss, mIoU, pixel accuracy."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    for ax, train_key, val_key, ylabel, title in [
        (axes[0], 'train_loss',      'val_loss',      'Loss',           'Loss'),
        (axes[1], 'train_miou',      'val_miou',      'Mean IoU',       'Mean IoU'),
        (axes[2], 'train_pixel_acc', 'val_pixel_acc', 'Pixel Accuracy', 'Pixel Accuracy'),
    ]:
        ax.plot(epochs, history[train_key], 'b-', label='Train')
        ax.plot(epochs, history[val_key],   'r-', label='Val')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{model_name} — {title}')
        ax.legend()

    plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('—', '-')
    plt.savefig(OUTPUT_DIR / f'{safe}_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


def visualize_predictions(model, dataset, indices, model_name='Model',
                          device=DEVICE, num_samples=8):
    """
    Display original image | ground truth mask | predicted mask for a
    selection of test indices.
    """
    model.eval()
    n           = min(num_samples, len(indices))
    fig, axes   = plt.subplots(n, 3, figsize=(18, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    denorm_mean = np.array([0.485, 0.456, 0.406])
    denorm_std  = np.array([0.229, 0.224, 0.225])

    for row, idx in enumerate(indices[:n]):
        img_tensor, mask_tensor = dataset[idx]
        img_np = img_tensor.numpy().transpose(1, 2, 0)
        img_np = (img_np * denorm_std + denorm_mean).clip(0, 1)

        with torch.no_grad():
            pred_mask = model(img_tensor.unsqueeze(0).to(device)
                              ).argmax(dim=1).squeeze(0).cpu().numpy()

        gt_mask = mask_tensor.numpy()

        gt_colored   = np.zeros((*gt_mask.shape,   3), dtype=np.uint8)
        pred_colored = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
        for c in range(NUM_CLASSES):
            gt_colored  [gt_mask   == c] = CLASS_COLORS[c]
            pred_colored[pred_mask == c] = CLASS_COLORS[c]

        for col, (im, title) in enumerate([
            (img_np,      'Input Image'),
            (gt_colored,  'Ground Truth'),
            (pred_colored,'Prediction'),
        ]):
            axes[row, col].imshow(im)
            axes[row, col].set_title(title)
            axes[row, col].axis('off')

    patches = [mpatches.Patch(color=np.array(CLASS_COLORS[i]) / 255.,
                              label=CLASS_NAMES[i])
               for i in range(NUM_CLASSES)]
    fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=10,
               bbox_to_anchor=(0.5, -0.01))
    plt.suptitle(f'{model_name} — Predictions', fontsize=14, y=1.01)
    plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('—', '-')
    plt.savefig(OUTPUT_DIR / f'{safe}_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix(cm, title='Confusion Matrix'):
    """Plot a row-normalised confusion matrix."""
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=13)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = cm_norm[i, j]
            if v > 0.005:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color='white' if v > 0.5 else 'black', fontsize=8)
    plt.tight_layout()
    return fig


print('Visualization utilities defined.')

Visualization utilities defined.


## 9. Grad-CAM Utilities

In [10]:
class SemanticSegmentationTarget:
    """
    Grad-CAM target for semantic segmentation.
    Aggregates logits for `category` across all spatial positions.
    """
    def __init__(self, category, mask=None):
        self.category = category
        self.mask     = mask

    def __call__(self, model_output):
        # model_output: (C, H, W)
        if self.mask is not None:
            return (model_output[self.category] * self.mask).sum()
        return model_output[self.category].sum()


def generate_gradcam_for_image(model, img_tensor, target_class, target_layer,
                               device=DEVICE):
    """
    Generate a Grad-CAM heatmap for one image and one target class.

    Args:
        model        : U-Net model (eval mode)
        img_tensor   : (3, H, W) normalised tensor
        target_class : class index 0-6
        target_layer : layer to hook (e.g. model.segmentation_head[0])

    Returns:
        cam_image     : (H, W, 3) uint8 overlay
        grayscale_cam : (H, W) float heatmap
    """
    input_tensor = img_tensor.unsqueeze(0).to(device)
    targets      = [SemanticSegmentationTarget(target_class)]

    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    denorm_mean = np.array([0.485, 0.456, 0.406])
    denorm_std  = np.array([0.229, 0.224, 0.225])
    img_np = img_tensor.numpy().transpose(1, 2, 0)
    img_np = (img_np * denorm_std + denorm_mean).clip(0, 1).astype(np.float32)

    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    return cam_image, grayscale_cam


def visualize_gradcam_gallery(model, dataset, indices, target_classes,
                              target_layer, model_name='Model', device=DEVICE):
    """
    Create a Grad-CAM gallery: rows = images, cols = target classes.
    First column is always the original input image.
    """
    n_imgs    = len(indices)
    n_classes = len(target_classes)

    fig, axes = plt.subplots(n_imgs, n_classes + 1,
                             figsize=(5 * (n_classes + 1), 5 * n_imgs))
    if n_imgs == 1:
        axes = axes[np.newaxis, :]

    denorm_mean = np.array([0.485, 0.456, 0.406])
    denorm_std  = np.array([0.229, 0.224, 0.225])

    for row, idx in enumerate(indices):
        img_tensor, _ = dataset[idx]
        img_np = img_tensor.numpy().transpose(1, 2, 0)
        img_np = (img_np * denorm_std + denorm_mean).clip(0, 1)

        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title('Input Image', fontsize=10)
        axes[row, 0].axis('off')

        for col, cls_idx in enumerate(target_classes):
            cam_img, _ = generate_gradcam_for_image(
                model, img_tensor, cls_idx, target_layer, device)
            axes[row, col + 1].imshow(cam_img)
            axes[row, col + 1].set_title(f'Grad-CAM: {CLASS_NAMES[cls_idx]}', fontsize=10)
            axes[row, col + 1].axis('off')

    plt.suptitle(f'{model_name} — Grad-CAM Visualization', fontsize=14, y=1.01)
    plt.tight_layout()
    safe = model_name.replace(' ', '_').replace('—', '-')
    plt.savefig(OUTPUT_DIR / f'{safe}_gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()


print('Grad-CAM utilities defined.')

Grad-CAM utilities defined.


## 10. Smoke Test — Both Architectures

In [11]:
# Load class weights computed in notebook 02
WEIGHTS_FILE = OUTPUT_DIR / 'class_weights.npy'
assert WEIGHTS_FILE.exists(), f'Run notebook 02 first — {WEIGHTS_FILE} not found.'

class_weights        = np.load(WEIGHTS_FILE)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f'Class weights loaded: {class_weights_tensor}')

criterion_smoke = CombinedLoss(class_weights=class_weights_tensor).to(DEVICE)

print('\n=== Smoke Test ===')
for enc_name, label in [('mobilenet_v2', 'MobileNetV2'), ('efficientnet-b0', 'EfficientNet-B0')]:
    model_test = create_unet_model(enc_name)
    model_test.eval()

    # Random batch: 2 images at IMG_SIZE
    dummy_imgs  = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    dummy_masks = torch.randint(0, NUM_CLASSES, (2, IMG_SIZE, IMG_SIZE)).to(DEVICE)

    with torch.no_grad():
        out  = model_test(dummy_imgs)
        loss = criterion_smoke(out, dummy_masks)

    assert out.shape  == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE), f'Unexpected output shape: {out.shape}'
    assert loss.item() > 0, 'Loss should be positive'

    print(f'  ✅  {label:<18}  output: {tuple(out.shape)}  loss: {loss.item():.4f}')
    del model_test
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print('\n✅ All smoke tests passed.')

Class weights loaded: tensor([0.0227, 1.5775, 1.2734, 0.6377, 1.1389, 1.8030, 0.5468])

=== Smoke Test ===
  ✅  MobileNetV2         output: (2, 7, 512, 512)  loss: 1.6544
  ✅  EfficientNet-B0     output: (2, 7, 512, 512)  loss: 3.4496

✅ All smoke tests passed.


## 11. Final Summary

In [12]:
print('=' * 65)
print('  NOTEBOOK 03 — COMPLETE')
print('=' * 65)
print('  Functions & classes ready for import in notebooks 04-06:')
print()
print('  Model')
print('    create_unet_model(encoder_name)')
print()
print('  Loss')
print('    CombinedLoss(class_weights, dice_weight, ce_weight)')
print()
print('  Metrics')
print('    compute_metrics(pred_masks, gt_masks)')
print('    print_metrics(metrics, title)')
print()
print('  Calibration')
print('    compute_calibration(pred_probs_list, gt_masks_list)')
print('    plot_reliability_diagram(cal_data, title, ax)')
print()
print('  Training')
print('    train_one_epoch(model, loader, criterion, optimizer)')
print('    evaluate(model, loader, criterion, return_predictions)')
print('    train_model(model, train_loader, val_loader, test_loader, ...)')
print()
print('  Visualisation')
print('    plot_training_history(history, model_name)')
print('    visualize_predictions(model, dataset, indices, model_name)')
print('    plot_confusion_matrix(cm, title)')
print()
print('  Grad-CAM')
print('    SemanticSegmentationTarget(category)')
print('    generate_gradcam_for_image(model, img_tensor, target_class, target_layer)')
print('    visualize_gradcam_gallery(model, dataset, indices, target_classes, target_layer)')
print()
print('  Next: run 04_step1_natural_background.ipynb')
print('=' * 65)

  NOTEBOOK 03 — COMPLETE
  Functions & classes ready for import in notebooks 04-06:

  Model
    create_unet_model(encoder_name)

  Loss
    CombinedLoss(class_weights, dice_weight, ce_weight)

  Metrics
    compute_metrics(pred_masks, gt_masks)
    print_metrics(metrics, title)

  Calibration
    compute_calibration(pred_probs_list, gt_masks_list)
    plot_reliability_diagram(cal_data, title, ax)

  Training
    train_one_epoch(model, loader, criterion, optimizer)
    evaluate(model, loader, criterion, return_predictions)
    train_model(model, train_loader, val_loader, test_loader, ...)

  Visualisation
    plot_training_history(history, model_name)
    visualize_predictions(model, dataset, indices, model_name)
    plot_confusion_matrix(cm, title)

  Grad-CAM
    SemanticSegmentationTarget(category)
    generate_gradcam_for_image(model, img_tensor, target_class, target_layer)
    visualize_gradcam_gallery(model, dataset, indices, target_classes, target_layer)

  Next: run 04_step1_na

In [13]:
!jupyter nbconvert --to html 03_model_and_training_utils.ipynb 

[NbConvertApp] Converting notebook 03_model_and_training_utils.ipynb to html
[NbConvertApp] Writing 423027 bytes to 03_model_and_training_utils.html
